In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_absolute_error, mean_squared_error, r2_score
from imblearn.over_sampling import SMOTE
import joblib
import matplotlib.pyplot as plt

df = pd.read_csv("titanic.csv")

X = df.drop(columns=['survived'])
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

numeric_features = ['age', 'sibsp', 'parch', 'fare']
categorical_features = ['sex', 'embarked']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(drop='first'))]), categorical_features)
    ]
)

clf_lr = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', LogisticRegression())])
clf_dt = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', DecisionTreeClassifier())])
clf_rf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier())])

clf_lr.fit(X_train, y_train)
clf_dt.fit(X_train, y_train)
clf_rf.fit(X_train, y_train)

for name, model in [('Logistic Regression', clf_lr), ('Decision Tree', clf_dt), ('Random Forest', clf_rf)]:
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    print(name)
    print(accuracy_score(y_test, preds))
    print(precision_score(y_test, preds))
    print(recall_score(y_test, preds))
    print(f1_score(y_test, preds))
    print(roc_auc_score(y_test, probs))

preprocessor.fit(X_train)
X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

smote = SMOTE()
X_train_sm, y_train_sm = smote.fit_resample(X_train_trans, y_train)
rf_smote = RandomForestClassifier()
rf_smote.fit(X_train_sm, y_train_sm)

param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10],
    'classifier__max_features': ['sqrt']
}
grid_rf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(oob_score=True))])
grid_search = GridSearchCV(grid_rf, param_grid, cv=3)
grid_search.fit(X_train, y_train)

reg_features = ['pclass', 'age', 'sibsp', 'parch']
X_reg = df[reg_features].fillna(df[reg_features].median())
y_reg = df['fare'].fillna(df['fare'].median())
X_r_train, X_r_test, y_r_train, y_r_test = train_test_split(X_reg, y_reg, test_size=0.2)

reg_model = LinearRegression()
reg_model.fit(X_r_train, y_r_train)
reg_preds = reg_model.predict(X_r_test)

print(mean_absolute_error(y_r_test, reg_preds))
print(np.sqrt(mean_squared_error(y_r_test, reg_preds)))
print(r2_score(y_r_test, reg_preds))
n = len(y_r_test)
p = X_r_test.shape[1]
r2 = r2_score(y_r_test, reg_preds)
adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))
print(adj_r2)

joblib.dump(clf_rf, 'best_pipeline.pkl')
loaded_pipeline = joblib.load('best_pipeline.pkl')
print(loaded_pipeline.predict(X_test.head(1)))

Logistic Regression
0.770949720670391
0.7692307692307693
0.5797101449275363
0.6611570247933884
0.7836627140974967
Decision Tree
0.7597765363128491
0.6805555555555556
0.7101449275362319
0.6950354609929078
0.7538866930171277
Random Forest
0.7988826815642458
0.7619047619047619
0.6956521739130435
0.7272727272727273
0.802042160737813
21.179088590617184
44.53885700710589
0.28027846268469103
0.26373313998778736
[0]
